# LeetCode Topic Discovery — KNN clustering on titles + LLM classification

**Goal.** Take the raw `Leetcode.csv` dump of ~3.6k problems and answer two questions:

1. *Unsupervised:* do the **titles alone** carry enough signal to group problems into
   meaningful families? We answer this with a **KNN / k-means pipeline where `k = √n`** —
   the classic rule-of-thumb for choosing the number of neighbours/clusters.
2. *Supervised-ish:* can a **generative model (via OpenRouter)** read a title and name the
   algorithmic **topic** — and how accurate is it when scored against the real LeetCode tags?

Finally we combine the two: use the clusters to **propagate a handful of LLM labels to the
whole dataset**, and measure how much labelling budget that saves.

---

### Pipeline

| Stage | What happens | Key object |
|---|---|---|
| 1 | Load + profile the CSV | `df` |
| 2 | Build ground truth from the `Topics` tags → 14 coarse topics | `gt_topics`, `gt_primary` |
| 3 | Normalise titles (lowercase, de-punctuate, strip roman/ordinal suffixes) | `clean_title` |
| 4 | Vectorise titles — TF-IDF over **words + character n-grams** | `X` |
| 5 | Choose **`k = √n`** | `K` |
| 6 | **KNN graph** (`NearestNeighbors`) — nearest-title lookup, sanity check | `knn` |
| 7 | **Cluster** with `KMeans(n_clusters=K)` | `cluster_id` |
| 8 | Inspect clusters: top terms, sample members, 2-D map | — |
| 9 | **LLM classification** through OpenRouter (batched, cached, threaded) | `llm_label` |
| 10 | Keyword baseline for comparison | `kw_label` |
| 11 | **Accuracy** — hit-rate, primary-label accuracy, macro-F1, confusion matrix | `scores` |
| 12 | Cluster ↔ label agreement (ARI / NMI / purity) | — |
| 13 | **KNN classifier** (`k = √n`) trained on LLM labels | — |
| 14 | Label-efficiency curve: how few LLM calls do we actually need? | — |

> **Cost note.** Stage 9 calls a hosted model. With the defaults below (`SAMPLE_SIZE = 800`,
> 25 titles per request) that is ~32 requests — cents, not dollars. Every answer is cached to
> disk, so re-running the notebook is free.

## 0. Setup

All tunable knobs live in this one cell so you never have to hunt through the notebook.

In [ ]:
# =============================================================================
# Imports
# =============================================================================
from __future__ import annotations

import json
import math
import os
import random
import re
import time
import warnings
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# =============================================================================
# Configuration — the only cell you should need to edit
# =============================================================================

# --- Reproducibility ---------------------------------------------------------
# k-means, t-SNE and the train/test split are all stochastic; pinning the seed
# means every number printed below is reproducible run-to-run.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Paths -------------------------------------------------------------------
CSV_PATH = Path("Leetcode.csv")          # input dataset
OUT_DIR = Path("outputs")                # everything we write lands here
OUT_DIR.mkdir(exist_ok=True)
CACHE_PATH = OUT_DIR / "llm_label_cache.json"   # title -> label, survives restarts

# --- LLM settings (OpenRouter) ----------------------------------------------
# OpenRouter exposes an OpenAI-compatible endpoint, so we drive it with the
# official `openai` SDK and just repoint `base_url`.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Any OpenRouter slug works. Cheap + strong at short classification tasks:
#   "anthropic/claude-haiku-4.5"   <- default
#   "openai/gpt-4o-mini"
#   "openai/gpt-4.1-mini"
#   "meta-llama/llama-3.3-70b-instruct"
#   "mistralai/mistral-small-3.2-24b-instruct"
MODEL = "anthropic/claude-haiku-4.5"

BATCH_SIZE = 25       # titles per request — batching is what keeps the cost down
MAX_WORKERS = 4       # parallel in-flight requests (raise if you have headroom)
MAX_RETRIES = 4       # per-batch retries with exponential backoff
TEMPERATURE = 0.0     # classification wants determinism, not creativity

# How many problems to send to the LLM.
#   800  -> ~32 requests, a few cents, plenty for every metric below
#   None -> label the entire dataset (~146 requests)
SAMPLE_SIZE = 800

USE_LLM = True        # set False to run the whole notebook offline on the baseline

# --- Display -----------------------------------------------------------------
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 170)

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print(f"model={MODEL} | batch={BATCH_SIZE} | sample={SAMPLE_SIZE} | seed={SEED}")

In [ ]:
# =============================================================================
# Plot theme — one place for colour + axis styling
# =============================================================================
# Colours come from a CVD-validated categorical palette. Slots are assigned in a
# FIXED order and never cycled: a chart with 4+ categories folds the tail into
# "Other" rather than inventing a 4th hue.

SURFACE = "#fcfcfb"      # chart background
INK = "#0b0b0b"          # primary text
INK_2 = "#52514e"        # secondary text (ticks, axis labels)
MUTED = "#b4b3ad"        # de-emphasised marks ("Other", context points)
GRID = "#e6e5e1"         # recessive gridlines

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]   # blue, orange, aqua — validated all-pairs

# Sequential blue ramp (light -> dark) for magnitude encodings such as heatmaps.
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue",
    ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"],
)

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "font.size": 10,
    "text.color": INK,
    "axes.labelcolor": INK_2,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "figure.dpi": 110,
})


def style_axes(ax, *, xgrid=False, ygrid=True):
    """Apply the recessive-chrome rules: no box, thin grid, muted ticks."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(colors=INK_2, length=0, labelsize=9)
    ax.grid(ygrid, axis="y", color=GRID, linewidth=0.8)
    ax.grid(xgrid, axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)          # data sits above the grid, never behind it
    return ax


def titled(ax, title, subtitle=None):
    """Left-aligned title + optional explanatory subtitle (the chart's caption).

    The pad has to clear the subtitle line, otherwise the two collide on axes
    that hide their tick labels.
    """
    ax.set_title(title, loc="left", pad=26 if subtitle else 10, color=INK)
    if subtitle:
        ax.text(0, 1.012, subtitle, transform=ax.transAxes,
                fontsize=9, color=INK_2, va="bottom")
    return ax

## 1. Load and profile the data

Before modelling anything, look at what we actually have: how many rows, what the columns
mean, and where the missing values are.

In [ ]:
# Read the raw dump. `Topics` is a comma-separated tag string; everything else is scalar.
df = pd.read_csv(CSV_PATH)

print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
print("columns:", list(df.columns), "\n")
df.head(8)

In [ ]:
# --- Where are the holes? ----------------------------------------------------
# `Example Test Cases` is entirely empty in this dump -> useless, drop it from consideration.
# `Topics` is missing on ~93 rows (mostly premium/Database problems); those rows can still be
# CLUSTERED (we only need the title) but cannot be SCORED (no ground truth to score against).
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0].to_string(), "\n")

print("Category breakdown:")
print(df["Category"].value_counts().to_string(), "\n")

print("Difficulty breakdown:")
print(df["Difficulty"].value_counts().to_string())

In [ ]:
# --- How rich is the tag vocabulary? -----------------------------------------
# Each problem carries 0..n tags. Counting them tells us how granular the raw label
# space is, and therefore how much we need to coarsen it before an LLM can hit it.
tag_counts = Counter()
for tag_string in df["Topics"].dropna():
    for tag in tag_string.split(","):
        tag_counts[tag.strip()] += 1

print(f"{len(tag_counts)} distinct raw tags across {df['Topics'].notna().sum():,} tagged problems\n")
print("Top 20 raw tags:")
for tag, n in tag_counts.most_common(20):
    print(f"  {n:5d}  {tag}")
print(f"\nLong tail: {sum(1 for n in tag_counts.values() if n < 20)} tags appear fewer than 20 times.")

## 2. Ground truth — collapse 72 raw tags into 14 coarse topics

The raw tag space is too fine-grained to score against: nobody, human or model, reliably
picks `Monotonic Queue` over `Sliding Window` from a title alone. So we map every raw tag
to one of **14 coarse topics** and score at that level.

Two ground-truth views come out of this, and they answer different questions:

- **`gt_topics`** — the *set* of coarse topics a problem legitimately belongs to.
  Used for **hit-rate**: "is the model's single answer one of the correct ones?" This is the
  fair metric, because a problem tagged `Array, Dynamic Programming` is genuinely both.
- **`gt_primary`** — one *canonical* topic per problem, chosen as the coarse topic of the
  problem's **rarest** raw tag. Rationale: `Array` is on 54% of problems and says almost
  nothing; `Segment Tree` is on 1.8% and says almost everything. The rarest tag is the most
  informative one. Used for the confusion matrix and macro-F1.

In [ ]:
# =============================================================================
# The taxonomy: 14 coarse topics, and the raw-tag -> coarse-topic mapping
# =============================================================================
# Each entry is (coarse topic -> the raw LeetCode tags that roll up into it).
# The one-line description is fed verbatim to the LLM, so the model and the scorer
# are working from the exact same definition. That matters: if the prompt and the
# ground truth disagree on what "Greedy" means, the accuracy number is meaningless.

TAXONOMY = {
    "Array & Matrix":               "Index/range manipulation of 1-D arrays, 2-D grids, prefix sums, sweeps.",
    "String":                       "Character-level processing, parsing, pattern/substring matching, tries.",
    "Hash Table & Counting":        "Frequency maps, lookup tables, counting/grouping by key.",
    "Two Pointers & Sliding Window":"A moving window or converging pointers over a sequence.",
    "Binary Search & Sorting":      "Sorting, order statistics, binary search over an answer or a sorted array.",
    "Dynamic Programming":          "Optimal substructure solved by memoisation or tabulation, incl. bitmask DP.",
    "Greedy":                       "Locally optimal choices, exchange arguments, interval scheduling.",
    "Recursion & Backtracking":     "Exhaustive search, permutations/combinations, divide-and-conquer, enumeration.",
    "Tree":                         "Binary trees, BSTs, tree traversal, segment trees, Fenwick/BIT.",
    "Graph":                        "Graph traversal (DFS/BFS), shortest paths, topological sort, union-find, MST.",
    "Stack, Queue & Linked List":   "Linear structures: stacks, queues, heaps/priority queues, linked lists.",
    "Math & Bit Manipulation":      "Number theory, combinatorics, geometry, probability, bit tricks, game theory.",
    "Design & Simulation":          "Implement a class/API, step-by-step simulation, data streams, concurrency.",
    "Database & Scripting":         "SQL queries and shell scripting problems (not algorithmic).",
}
TOPICS = list(TAXONOMY)

TAG_TO_TOPIC = {
    # -- Array & Matrix
    "Array": "Array & Matrix", "Matrix": "Array & Matrix",
    "Prefix Sum": "Array & Matrix", "Line Sweep": "Array & Matrix",
    # -- String
    "String": "String", "String Matching": "String", "Rolling Hash": "String",
    "Suffix Array": "String", "Trie": "String",
    # -- Hash Table & Counting
    "Hash Table": "Hash Table & Counting", "Counting": "Hash Table & Counting",
    "Hash Function": "Hash Table & Counting",
    # -- Two Pointers & Sliding Window
    "Two Pointers": "Two Pointers & Sliding Window",
    "Sliding Window": "Two Pointers & Sliding Window",
    # -- Binary Search & Sorting
    "Binary Search": "Binary Search & Sorting", "Sorting": "Binary Search & Sorting",
    "Sort": "Binary Search & Sorting", "Merge Sort": "Binary Search & Sorting",
    "Counting Sort": "Binary Search & Sorting", "Bucket Sort": "Binary Search & Sorting",
    "Radix Sort": "Binary Search & Sorting", "Quickselect": "Binary Search & Sorting",
    "Ordered Set": "Binary Search & Sorting",
    # -- Dynamic Programming
    "Dynamic Programming": "Dynamic Programming", "Memoization": "Dynamic Programming",
    "Bitmask": "Dynamic Programming",
    # -- Greedy
    "Greedy": "Greedy",
    # -- Recursion & Backtracking
    "Backtracking": "Recursion & Backtracking", "Recursion": "Recursion & Backtracking",
    "Divide and Conquer": "Recursion & Backtracking", "Enumeration": "Recursion & Backtracking",
    # -- Tree
    "Tree": "Tree", "Binary Tree": "Tree", "Binary Search Tree": "Tree",
    "Segment Tree": "Tree", "Binary Indexed Tree": "Tree",
    # -- Graph
    "Graph": "Graph", "Depth-First Search": "Graph", "Breadth-First Search": "Graph",
    "Topological Sort": "Graph", "Shortest Path": "Graph", "Union Find": "Graph",
    "Minimum Spanning Tree": "Graph", "Strongly Connected Component": "Graph",
    "Eulerian Circuit": "Graph", "Biconnected Component": "Graph",
    # -- Stack, Queue & Linked List
    "Stack": "Stack, Queue & Linked List", "Queue": "Stack, Queue & Linked List",
    "Monotonic Stack": "Stack, Queue & Linked List", "Monotonic Queue": "Stack, Queue & Linked List",
    "Linked List": "Stack, Queue & Linked List", "Doubly-Linked List": "Stack, Queue & Linked List",
    "Heap (Priority Queue)": "Stack, Queue & Linked List", "Iterator": "Stack, Queue & Linked List",
    # -- Math & Bit Manipulation
    "Math": "Math & Bit Manipulation", "Bit Manipulation": "Math & Bit Manipulation",
    "Number Theory": "Math & Bit Manipulation", "Combinatorics": "Math & Bit Manipulation",
    "Geometry": "Math & Bit Manipulation", "Probability and Statistics": "Math & Bit Manipulation",
    "Randomized": "Math & Bit Manipulation", "Reservoir Sampling": "Math & Bit Manipulation",
    "Rejection Sampling": "Math & Bit Manipulation", "Game Theory": "Math & Bit Manipulation",
    "Brainteaser": "Math & Bit Manipulation",
    # -- Design & Simulation
    "Design": "Design & Simulation", "Simulation": "Design & Simulation",
    "Data Stream": "Design & Simulation", "Interactive": "Design & Simulation",
    "Concurrency": "Design & Simulation",
    # -- Database & Scripting
    "Database": "Database & Scripting", "Shell": "Database & Scripting",
}

# Guard rail: if LeetCode ever adds a tag, fail loudly here rather than silently
# dropping problems out of the ground truth.
unmapped = set(tag_counts) - set(TAG_TO_TOPIC)
assert not unmapped, f"Unmapped tags — extend TAG_TO_TOPIC: {sorted(unmapped)}"
print(f"{len(TAG_TO_TOPIC)} raw tags -> {len(TOPICS)} coarse topics. Full coverage.")

In [ ]:
# =============================================================================
# Build the two ground-truth views
# =============================================================================
# `tag_counts` (computed above) doubles as our rarity table: the LOWER the count,
# the more specific — and therefore more informative — the tag.

def parse_tags(tag_string: str) -> list[str]:
    """'Array, Two Pointers' -> ['Array', 'Two Pointers']. NaN/empty -> []."""
    if not isinstance(tag_string, str) or not tag_string.strip():
        return []
    return [t.strip() for t in tag_string.split(",") if t.strip()]


def coarse_topic_set(tag_string: str) -> frozenset[str]:
    """All coarse topics a problem legitimately belongs to (may be several)."""
    return frozenset(TAG_TO_TOPIC[t] for t in parse_tags(tag_string))


def primary_topic(tag_string: str) -> str | None:
    """Canonical single label = coarse topic of the problem's RAREST raw tag.

    'Array, Dynamic Programming' -> Array appears 1973x, DP appears 607x,
    so DP wins and the problem is canonically 'Dynamic Programming'.
    """
    tags = parse_tags(tag_string)
    if not tags:
        return None
    rarest = min(tags, key=lambda t: tag_counts[t])
    return TAG_TO_TOPIC[rarest]


df["gt_topics"] = df["Topics"].map(coarse_topic_set)      # frozenset per row
df["gt_primary"] = df["Topics"].map(primary_topic)        # str or None
df["n_tags"] = df["Topics"].map(lambda s: len(parse_tags(s)))

# `labelled` = the rows we can actually score against. Clustering still uses ALL rows.
labelled = df[df["gt_primary"].notna()].copy()

print(f"scoreable rows: {len(labelled):,} of {len(df):,} "
      f"({len(labelled)/len(df):.1%})")
print(f"avg raw tags per scoreable problem: {labelled['n_tags'].mean():.2f}")
print(f"avg coarse topics per problem: {labelled['gt_topics'].map(len).mean():.2f}\n")

print("Primary-topic distribution (the canonical single label):")
print(labelled["gt_primary"].value_counts().to_string())

## 3. Normalise the titles

The clustering signal is the title and nothing else, so it is worth cleaning carefully.

Two title quirks matter:

- **Sequel suffixes.** `Earliest Finish Time for Land and Water Rides I` and `... II` are the
  same problem family. Stripping the trailing roman numeral pulls them together.
- **Filler words.** `of`, `the`, `a` appear everywhere and carry no topical signal — TF-IDF
  down-weights them automatically, but removing them makes the cluster top-terms readable.

In [ ]:
# Domain stopwords: English filler plus LeetCode-boilerplate verbs that appear in
# hundreds of titles and so cannot discriminate between topics.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "into",
    "is", "it", "of", "on", "or", "that", "the", "to", "with", "without", "all",
    "any", "can", "do", "does", "each", "given", "if", "its", "not", "one", "so",
    "such", "than", "then", "there", "these", "they", "this", "was", "we", "you",
}

# Trailing sequel markers: "Problem II", "Problem 2", "Problem III" -> "Problem".
SEQUEL_RE = re.compile(r"\s+(?:i{1,3}|iv|v|vi{1,3}|ix|x|\d{1,2})$", re.IGNORECASE)


def clean_title(title: str) -> str:
    """Lowercase, drop punctuation/digits, strip sequel suffix and stopwords."""
    t = title.lower()
    t = SEQUEL_RE.sub("", t)                  # drop the sequel marker FIRST...
    t = re.sub(r"[^a-z\s]", " ", t)           # ...then flatten punctuation + digits
    tokens = [w for w in t.split() if w not in STOPWORDS and len(w) > 1]
    return " ".join(tokens)


df["clean_title"] = df["Title"].map(clean_title)

# Eyeball a few transformations — cheap insurance against a regex that over-strips.
preview = df.sample(8, random_state=SEED)[["Title", "clean_title"]]
print("Title normalisation samples:\n")
for original, cleaned in preview.itertuples(index=False):
    print(f"  {original[:58]:<58} -> {cleaned}")

empty = (df["clean_title"].str.len() == 0).sum()
print(f"\ntitles reduced to empty string: {empty} (these fall back to raw text below)")

## 4. Vectorise — TF-IDF over words **and** characters

A single word-level TF-IDF is brittle on short text: `Subarray` and `Subarrays` become
unrelated dimensions, and a 4-word title gives you 4 non-zero features to work with.

So we take the union of two views and let them complement each other:

| View | `analyzer` | Catches |
|---|---|---|
| Word (1–2 grams) | `word` | Topical phrases: `binary tree`, `linked list`, `prefix sum` |
| Char (3–5 grams) | `char_wb` | Morphology + typos: `subarray` ≈ `subarrays` ≈ `sub-array` |

Both are L2-normalised, so cosine similarity is just a dot product — which is what makes the
KNN and k-means steps below cheap.

In [ ]:
# `sublinear_tf=True` applies 1+log(tf): a word appearing twice in a 5-word title
# should not count double. `min_df=2` drops hapax terms that only ever match one title.
word_tfidf = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2,
    sublinear_tf=True, strip_accents="unicode",
)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=3,
    sublinear_tf=True, strip_accents="unicode",
)

vectorizer = FeatureUnion([("word", word_tfidf), ("char", char_tfidf)])

# Fit on EVERY title (including untagged ones) — clustering is unsupervised, so
# unlabelled rows are free extra structure rather than a leak.
X = vectorizer.fit_transform(df["clean_title"])
X = normalize(X)          # L2 -> cosine distance == euclidean distance on the unit sphere

n_word = len(word_tfidf.vocabulary_)
n_char = len(char_tfidf.vocabulary_)
print(f"documents: {X.shape[0]:,}")
print(f"features : {X.shape[1]:,}  ({n_word:,} word n-grams + {n_char:,} char n-grams)")
print(f"density  : {X.nnz / (X.shape[0] * X.shape[1]):.4%} (sparse, as expected for short text)")
print(f"avg non-zero features per title: {X.nnz / X.shape[0]:.0f}")

## 5. Choose `k = √n`

The rule of thumb: with `n` points, use `k ≈ √n`. It is a heuristic, not a theorem, but it is
a principled starting point in both roles it plays here —

- as a **neighbourhood size** it grows slowly enough that neighbourhoods stay local as `n` grows;
- as a **cluster count** it puts roughly `√n` points in each of `√n` clusters, which is the
  balance point between "one giant cluster" and "one cluster per point".

We use the *same* `k` for the KNN graph, k-means, and the KNN classifier, so the whole
notebook is driven by a single hyperparameter.

In [ ]:
N = X.shape[0]
K = int(round(math.sqrt(N)))

print(f"n = {N:,}")
print(f"k = round(sqrt(n)) = {K}")
print(f"-> expected average cluster size = n / k = {N / K:.1f} problems")

## 6. The KNN graph

Before clustering, use plain **k-nearest-neighbours** to sanity-check the vector space: pick a
few titles and look at what lands closest. If the neighbours are obviously related problems,
the representation is good and k-means has something to work with. If they look random, no
amount of clustering will save us.

In [ ]:
# Brute force over 3.6k x ~40k sparse vectors is fast and exact — no need for an
# approximate index at this scale. metric="cosine" matches our L2-normalised TF-IDF.
knn = NearestNeighbors(n_neighbors=K, metric="cosine", algorithm="brute")
knn.fit(X)

# distances[i, j] / indices[i, j] = the j-th nearest neighbour of row i.
distances, indices = knn.kneighbors(X)

print(f"KNN graph built: {N:,} nodes, k = {K} neighbours each")
print(f"mean cosine distance to nearest neighbour : {distances[:, 1].mean():.3f}")
print(f"mean cosine distance to {K}-th neighbour     : {distances[:, -1].mean():.3f}")

In [ ]:
def show_neighbours(query_index: int, n: int = 6) -> None:
    """Print a title and its n closest titles, with cosine distance."""
    print(f"\nQUERY  {df['Title'].iloc[query_index]}")
    print(f"       tags: {df['Topics'].iloc[query_index]}")
    # Column 0 is the point itself (distance 0), so start at 1.
    for rank, (neighbour, dist) in enumerate(
        zip(indices[query_index, 1:n + 1], distances[query_index, 1:n + 1]), start=1
    ):
        print(f"  {rank}. [{dist:.3f}] {df['Title'].iloc[neighbour]}")


# Probe a few deliberately different corners of the dataset.
for probe in ["Binary Tree Inorder Traversal", "Longest Palindromic Substring",
              "Number of Islands", "Design Twitter"]:
    matches = df.index[df["Title"] == probe]
    if len(matches):
        show_neighbours(df.index.get_loc(matches[0]))

## 7. Cluster with `KMeans(n_clusters=k)`

KNN gives us local neighbourhoods; k-means turns those into a hard partition. On L2-normalised
TF-IDF, euclidean k-means is equivalent to **spherical k-means** (cosine), which is the right
geometry for text.

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=5, max_iter=300, random_state=SEED)
df["cluster_id"] = kmeans.fit_predict(X)

sizes = df["cluster_id"].value_counts()

# Silhouette on a 1k subsample — the full O(n^2) pairwise matrix is unnecessary here
# and the subsampled estimate is stable at this size.
sil = silhouette_score(X, df["cluster_id"], metric="cosine",
                       sample_size=1000, random_state=SEED)

print(f"clusters      : {K}")
print(f"inertia       : {kmeans.inertia_:,.1f}")
print(f"silhouette    : {sil:.3f}   (short-text TF-IDF sits low by nature; "
      f"> 0 means structure exists)")
print(f"\ncluster sizes : min={sizes.min()}  median={int(sizes.median())}  "
      f"max={sizes.max()}  mean={sizes.mean():.1f}")
print(f"singleton clusters: {(sizes == 1).sum()}")

In [ ]:
# --- Chart: how evenly did the partition split? ------------------------------
# Form choice: ranked magnitude across many categories -> horizontal bars, sorted.
# One series, so no legend is needed; the title names the measure.
top = sizes.head(25)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
y = np.arange(len(top))

ax.barh(y, top.values, height=0.62, color=SERIES[0])
ax.set_yticks(y)
ax.set_yticklabels([f"cluster {c}" for c in top.index])
ax.invert_yaxis()                       # largest at the top
ax.axvline(N / K, color=SERIES[1], linewidth=2, zorder=3)
ax.text(N / K, len(top) - 0.2, f"  even split = {N/K:.0f}",
        color=SERIES[1], fontsize=9, va="bottom")

# Direct-label every bar — 25 marks is few enough that labels beat an x-axis lookup.
# The offset is scaled to the axis range so short bars' labels clear the reference line.
gap = top.max() * 0.012
for yi, v in zip(y, top.values):
    ax.text(v + gap, yi, str(v), va="center", fontsize=8.5, color=INK_2)
ax.set_xlim(0, top.max() * 1.09)

style_axes(ax, xgrid=True, ygrid=False)
ax.set_xlabel("problems in cluster")
titled(ax, "25 largest title clusters",
       f"k-means, k = sqrt(n) = {K} · {N:,} problems · sizes are uneven by design")
plt.tight_layout()
plt.show()

## 8. Inspect the clusters

Metrics tell you *whether* the clustering worked; reading the clusters tells you *how*. For
each big cluster we print its centroid's heaviest terms (the theme), a few member titles, and
the dominant real tag — which is a preview of the purity number in section 12.

In [ ]:
from functools import lru_cache

WORD_FEATURE_NAMES = word_tfidf.get_feature_names_out()   # hoisted: rebuilding this
                                                          # per call is O(vocab) each time


@lru_cache(maxsize=None)
def cluster_top_terms(cluster: int, n_terms: int = 6) -> tuple[str, ...]:
    """The n highest-weighted WORD features at this cluster's centroid.

    We read the centroid directly rather than re-scoring members: the centroid IS
    the cluster's average document, so its heaviest terms are its theme.
    Cached because section 15 calls this once per row.
    """
    centroid = kmeans.cluster_centers_[cluster][:n_word]     # word block only
    return tuple(WORD_FEATURE_NAMES[i] for i in centroid.argsort()[::-1][:n_terms])


# Show the 12 biggest clusters: theme terms, a few members, and the dominant real tag.
print("=" * 100)
for cluster in sizes.head(12).index:
    members = df[df["cluster_id"] == cluster]
    real = Counter(t for s in members["gt_primary"].dropna() for t in [s])
    dominant = real.most_common(1)[0] if real else ("n/a", 0)
    purity = dominant[1] / max(len(members.dropna(subset=["gt_primary"])), 1)

    print(f"\nCLUSTER {cluster}  ({len(members)} problems)")
    print(f"  theme terms   : {', '.join(cluster_top_terms(cluster))}")
    print(f"  dominant tag  : {dominant[0]} ({purity:.0%} of tagged members)")
    print(f"  members       :")
    for title in members["Title"].head(5):
        print(f"      - {title}")
print("\n" + "=" * 100)

In [ ]:
# --- Chart: 2-D map of the corpus --------------------------------------------
# t-SNE on a 50-D SVD projection. SVD first because t-SNE on 40k sparse dims is
# both slow and noisy; 50 components retains the bulk of the variance cheaply.
svd = TruncatedSVD(n_components=50, random_state=SEED)
X_svd = svd.fit_transform(X)
print(f"SVD(50) explains {svd.explained_variance_ratio_.sum():.1%} of variance")

tsne = TSNE(n_components=2, perplexity=30, metric="cosine", init="pca",
            learning_rate="auto", max_iter=1000, random_state=SEED)
XY = tsne.fit_transform(X_svd)
df["tsne_x"], df["tsne_y"] = XY[:, 0], XY[:, 1]
print("t-SNE embedding ready")

In [ ]:
# Colour by GROUND TRUTH, not by cluster id: the question this chart answers is
# "do the title clusters line up with real topics?", and cluster ids are arbitrary.
#
# 14 topics is far too many for categorical colour, so we keep the top 3 by volume
# and fold everything else into a muted "Other" — the palette's fixed 3-slot,
# all-pairs-validated head. Never invent a 4th hue.
top3 = labelled["gt_primary"].value_counts().head(3).index.tolist()

fig, ax = plt.subplots(figsize=(8.5, 7))

# Untagged + non-top-3 problems form the recessive backdrop.
rest = df[~df["gt_primary"].isin(top3)]
ax.scatter(rest["tsne_x"], rest["tsne_y"], s=6, c=MUTED, alpha=0.45,
           linewidths=0, label=f"Other / untagged ({len(rest):,})")

for topic, colour in zip(top3, SERIES):
    sub = df[df["gt_primary"] == topic]
    ax.scatter(sub["tsne_x"], sub["tsne_y"], s=11, c=colour, alpha=0.85,
               linewidths=0.5, edgecolors=SURFACE,     # 2px-equivalent surface ring
               label=f"{topic} ({len(sub):,})")

style_axes(ax, xgrid=False, ygrid=False)
ax.set_xticks([]); ax.set_yticks([])
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
legend = ax.legend(loc="upper left", frameon=False, fontsize=9, markerscale=1.6)
for text in legend.get_texts():
    text.set_color(INK_2)
titled(ax, "Title space, coloured by true primary topic",
       "t-SNE of TF-IDF titles · contiguous colour = titles alone separate the topic")
plt.tight_layout()
plt.show()

## 9. A keyword baseline to beat

An accuracy number means nothing on its own — 62% is impressive or embarrassing depending on
what a trivial method scores. So before spending a cent on inference, build the dumbest
reasonable classifier: match keywords in the title, fall back to the majority class.

This is the floor. If the LLM cannot clear it, it is not earning its API bill. It also doubles
as the **offline fallback** — with no API key the notebook substitutes these labels so every
cell below still runs.

In [ ]:
# Hand-written keyword rules. Ordered most-specific first: "binary tree" must be
# checked before "tree", or the more specific rule never fires.
KEYWORD_RULES = [
    ("Database & Scripting",          ["sql", "query", "table", "employee", "department", "salary", "customer", "orders", "report"]),
    ("Tree",                          ["binary tree", "bst", "tree", "root", "leaf", "ancestor", "subtree"]),
    ("Graph",                         ["graph", "island", "node", "path between", "network", "connected", "course", "city", "cities", "route"]),
    ("Stack, Queue & Linked List",    ["linked list", "stack", "queue", "heap", "priority"]),
    ("String",                        ["string", "substring", "palindrom", "word", "letter", "character", "anagram", "vowel", "prefix"]),
    ("Two Pointers & Sliding Window", ["window", "two pointer", "consecutive"]),
    ("Binary Search & Sorting",       ["sort", "kth", "median", "search in", "smallest", "largest"]),
    ("Dynamic Programming",           ["longest", "minimum cost", "maximum profit", "ways to", "number of ways"]),
    ("Design & Simulation",           ["design", "implement", "simulat", "iterator", "cache", "system"]),
    ("Math & Bit Manipulation",       ["bit", "xor", "prime", "digit", "modulo", "power of", "sum of squares", "probability", "game"]),
    ("Hash Table & Counting",         ["count", "frequency", "duplicate", "unique", "occurrence"]),
    ("Greedy",                        ["minimum number of", "maximum number of", "fewest"]),
    ("Array & Matrix",                ["matrix", "grid", "subarray", "array", "interval", "range"]),
]

# The trivial floor beneath even the keyword rules: always guess the biggest class.
MAJORITY_CLASS = labelled["gt_primary"].value_counts().idxmax()


def keyword_label(title: str) -> str:
    """First matching rule wins; otherwise predict the majority class."""
    lowered = title.lower()
    for topic, keywords in KEYWORD_RULES:
        if any(kw in lowered for kw in keywords):
            return topic
    return MAJORITY_CLASS


baseline_all = labelled["Title"].map(keyword_label)
baseline_hit = np.mean([p in s for p, s in zip(baseline_all, labelled["gt_topics"])])

print(f"majority-class fallback : {MAJORITY_CLASS} "
      f"({(labelled['gt_primary'] == MAJORITY_CLASS).mean():.1%} of the corpus)")
print(f"keyword hit rate        : {baseline_hit:.1%} over all {len(labelled):,} tagged problems")
print(f"rules fired on          : {(baseline_all != MAJORITY_CLASS).mean():.1%} of titles\n")
print("Keyword baseline distribution:")
print(baseline_all.value_counts().to_string())

## 10. Generative classification through OpenRouter

Now the supervised half. We hand the model a **title only** — no tags, no description, no
difficulty — and ask it to pick one of our 14 topics.

Design decisions that matter for cost and correctness:

| Decision | Why |
|---|---|
| **Batch 25 titles per request** | One system prompt amortised across 25 classifications — ~25× cheaper than one call each. |
| **Numbered items + JSON out** | Indices make the response impossible to misalign, even if the model reorders or drops one. |
| **`temperature=0`** | Classification wants the mode, not a sample. |
| **Disk cache keyed by title** | Re-running the notebook costs nothing; a crashed run resumes where it stopped. |
| **Thread pool + backoff** | Latency-bound work; 4 workers is polite and roughly 4× faster. |

**API key.** Set `OPENROUTER_API_KEY` in your environment or in a `.env` file next to this
notebook. If no key is found the notebook degrades gracefully to the keyword baseline in
section 10, so everything below still runs.

In [ ]:
# =============================================================================
# Client setup
# =============================================================================
from openai import OpenAI     # OpenRouter speaks the OpenAI wire protocol


def get_api_key() -> str | None:
    """Look for the key in the environment, then in a local .env file."""
    key = os.environ.get("OPENROUTER_API_KEY")
    if key:
        return key.strip()
    for env_file in (Path(".env"), Path("../.env")):
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                if line.startswith("OPENROUTER_API_KEY"):
                    return line.split("=", 1)[1].strip().strip("\"'")
    return None


API_KEY = get_api_key()
client = None

if USE_LLM and API_KEY:
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=API_KEY)
    print(f"OpenRouter client ready -> {MODEL}")
else:
    USE_LLM = False
    print("No OPENROUTER_API_KEY found (or USE_LLM=False).")
    print("Falling back to the keyword baseline; every cell below still runs.")

# Every chart title and results table names this, NOT `MODEL` — so an offline run
# never reports keyword-rule numbers under the model's name.
LABEL_SOURCE = MODEL if USE_LLM else "keyword baseline (OFFLINE — no API key)"

In [ ]:
# =============================================================================
# Prompt construction
# =============================================================================
# The taxonomy is injected verbatim from TAXONOMY, so the model is scored against
# exactly the definitions it was given. Never let the prompt and the scorer drift.

TAXONOMY_BLOCK = "\n".join(f"- {name}: {desc}" for name, desc in TAXONOMY.items())

SYSTEM_PROMPT = f"""You are an expert competitive-programming coach who labels LeetCode \
problems by the single PRIMARY algorithmic technique required to solve them.

You will be given a numbered list of problem TITLES only. For each title, infer the most \
likely primary technique and assign exactly one label from this fixed taxonomy:

{TAXONOMY_BLOCK}

Rules:
- Choose exactly one label per title, copied EXACTLY as written above.
- Judge by the technique needed to SOLVE it, not by the nouns in the title. \
"Number of Islands" is Graph (flood fill), not Array & Matrix.
- If a title mentions SQL/table/query semantics, it is Database & Scripting.
- Titles are terse and ambiguous; commit to your best guess rather than hedging.

Respond with ONLY a JSON object, no prose, no markdown fence:
{{"labels": [{{"id": 0, "label": "..."}}, {{"id": 1, "label": "..."}}]}}"""


def build_user_prompt(titles: list[str]) -> str:
    """Number the batch so responses can be realigned by id, not by position."""
    listing = "\n".join(f"{i}. {t}" for i, t in enumerate(titles))
    return f"Classify these {len(titles)} LeetCode problem titles:\n\n{listing}"


print(SYSTEM_PROMPT[:600] + "\n...\n")
print("--- example user message ---")
print(build_user_prompt(df["Title"].head(3).tolist()))

In [ ]:
# =============================================================================
# Robust response parsing + a batched, retrying, cached classifier
# =============================================================================

# Lowercase lookup so a model that returns "greedy" instead of "Greedy" still lands.
_TOPIC_LOOKUP = {t.lower(): t for t in TOPICS}
UNKNOWN = "UNKNOWN"      # explicit bucket — never silently guess on the model's behalf


def canonicalise(raw_label: str) -> str:
    """Snap a model-returned string onto the taxonomy, or return UNKNOWN."""
    if not isinstance(raw_label, str):
        return UNKNOWN
    cleaned = raw_label.strip().strip(".")
    if cleaned.lower() in _TOPIC_LOOKUP:
        return _TOPIC_LOOKUP[cleaned.lower()]
    # Partial match: the model wrote "Graph traversal" or "DP (Dynamic Programming)".
    for topic in TOPICS:
        if topic.lower() in cleaned.lower() or cleaned.lower() in topic.lower():
            return topic
    return UNKNOWN


def extract_json(text: str) -> dict:
    """Pull the JSON object out of a response that may be fenced or chatty."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, re.DOTALL)      # greedy: outermost braces
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return {}


def classify_batch(titles: list[str]) -> list[str]:
    """Classify one batch. Returns a label per input title, aligned by index."""
    labels = [UNKNOWN] * len(titles)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=40 * len(titles) + 200,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(titles)},
                ],
                extra_headers={"X-Title": "leetcode-topic-clustering"},
            )
            payload = extract_json(response.choices[0].message.content or "")
            for item in payload.get("labels", []):
                idx = item.get("id")
                if isinstance(idx, int) and 0 <= idx < len(titles):
                    labels[idx] = canonicalise(item.get("label", ""))

            # Accept the batch once most items came back valid; a stray UNKNOWN
            # is cheaper to live with than another full round-trip.
            if sum(l != UNKNOWN for l in labels) >= 0.8 * len(titles):
                return labels
        except Exception as exc:
            if attempt == MAX_RETRIES - 1:
                print(f"  batch failed after {MAX_RETRIES} attempts: {type(exc).__name__}: {exc}")
                return labels
        time.sleep(2 ** attempt)      # 1s, 2s, 4s, 8s exponential backoff

    return labels


def load_cache() -> dict[str, str]:
    """Cache is keyed by (model, title) so switching models does not read stale labels."""
    if CACHE_PATH.exists():
        return json.loads(CACHE_PATH.read_text())
    return {}


def save_cache(cache: dict[str, str]) -> None:
    CACHE_PATH.write_text(json.dumps(cache, indent=0))


def classify_titles(titles: list[str]) -> dict[str, str]:
    """Classify many titles: cache-aware, batched, threaded, incremental-save."""
    if not USE_LLM:
        # No key / LLM disabled -> substitute the keyword baseline so the rest of the
        # notebook still executes. Say so loudly: the "LLM" rows below are NOT an LLM.
        print("!" * 74)
        print("  OFFLINE FALLBACK — no API key, using keyword labels as a stand-in.")
        print("  Every 'LLM' number below is really the baseline. Set OPENROUTER_API_KEY")
        print("  and re-run this cell for the real comparison.")
        print("!" * 74)
        return {t: keyword_label(t) for t in titles}

    cache = load_cache()
    todo = [t for t in dict.fromkeys(titles) if f"{MODEL}||{t}" not in cache]

    print(f"{len(titles):,} requested · {len(titles) - len(todo):,} cached · {len(todo):,} to fetch")
    if not todo:
        return {t: cache.get(f"{MODEL}||{t}", UNKNOWN) for t in titles}

    batches = [todo[i:i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    print(f"-> {len(batches)} requests of <= {BATCH_SIZE} titles, {MAX_WORKERS} in parallel\n")

    start = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(classify_batch, b): b for b in batches}
        for future in as_completed(futures):
            batch = futures[future]
            for title, label in zip(batch, future.result()):
                cache[f"{MODEL}||{title}"] = label
            done += 1
            if done % 5 == 0 or done == len(batches):
                save_cache(cache)      # checkpoint: a crash never loses more than 5 batches
                print(f"  {done}/{len(batches)} batches · {time.time() - start:.0f}s elapsed")

    save_cache(cache)
    return {t: cache.get(f"{MODEL}||{t}", UNKNOWN) for t in titles}


print("classifier ready")

In [ ]:
# =============================================================================
# Pick the evaluation sample — stratified by cluster
# =============================================================================
# A uniform random sample would over-represent the big clusters and leave small ones
# with zero labels, which would break the cluster-propagation analysis in section 14.
# Instead we take an equal quota from every cluster.

eval_pool = df[df["gt_primary"].notna()]      # only rows we can score

def stratified_sample(frame: pd.DataFrame, per_group: int, by: str = "cluster_id") -> pd.DataFrame:
    """Take up to `per_group` rows from each group, preserving all columns.

    Written as an explicit loop rather than `groupby().apply()`: pandas 3.0 consumes
    the grouping column inside `apply`, so `cluster_id` would vanish from the result.
    """
    parts = [g.sample(min(len(g), per_group), random_state=SEED)
             for _, g in frame.groupby(by)]
    return pd.concat(parts).copy()


if SAMPLE_SIZE is None or SAMPLE_SIZE >= len(eval_pool):
    sample = eval_pool.copy()
    print(f"labelling ALL {len(sample):,} scoreable problems")
else:
    per_cluster = math.ceil(SAMPLE_SIZE / K)
    sample = stratified_sample(eval_pool, per_cluster)
    print(f"stratified sample: <= {per_cluster} per cluster x {K} clusters "
          f"-> {len(sample):,} problems")

print(f"clusters represented: {sample['cluster_id'].nunique()} / {K}")
print(f"topics represented  : {sample['gt_primary'].nunique()} / {len(TOPICS)}")

In [ ]:
# =============================================================================
# Run the classification
# =============================================================================
label_map = classify_titles(sample["Title"].tolist())
sample["llm_label"] = sample["Title"].map(label_map)

n_unknown = (sample["llm_label"] == UNKNOWN).sum()
print(f"\nlabelled: {len(sample) - n_unknown:,} · unresolved: {n_unknown}")
print("\nLLM label distribution:")
print(sample["llm_label"].value_counts().to_string())

## 11. Accuracy

Three numbers, measuring three different things. Read them together.

| Metric | Definition | Why it matters |
|---|---|---|
| **Hit rate** | prediction ∈ `gt_topics` (the *set* of valid topics) | The fair metric. A problem tagged `Array, DP` is genuinely both; punishing the model for picking either is wrong. |
| **Primary accuracy** | prediction == `gt_primary` (rarest-tag canonical label) | The strict metric. Forces the model to find the *distinguishing* technique, not the obvious one. |
| **Macro-F1** | unweighted mean F1 across topics | Guards against the majority-class trap: `Array & Matrix` is 30%+ of the data, so a lazy model scores well on accuracy alone. |

The **all-majority** and **keyword** rows are the floors the LLM has to clear.

In [ ]:
def evaluate(predictions: pd.Series, name: str) -> dict:
    """Score a prediction column against both ground-truth views."""
    truth_sets = sample["gt_topics"]
    truth_primary = sample["gt_primary"]

    hit_rate = np.mean([p in s for p, s in zip(predictions, truth_sets)])
    primary_acc = accuracy_score(truth_primary, predictions)
    macro_f1 = f1_score(truth_primary, predictions, average="macro", zero_division=0)
    weighted_f1 = f1_score(truth_primary, predictions, average="weighted", zero_division=0)

    return {
        "method": name,
        "hit_rate": hit_rate,
        "primary_acc": primary_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "coverage": float(np.mean(predictions != UNKNOWN)),
    }


# Score the keyword baseline on exactly the same rows as the LLM — comparing them
# on different samples would make the delta meaningless.
sample["kw_label"] = sample["Title"].map(keyword_label)

# Two floors to clear, then the real contender.
always_majority = pd.Series([MAJORITY_CLASS] * len(sample), index=sample.index)

scores = pd.DataFrame([
    evaluate(always_majority, "all-majority (trivial floor)"),
    evaluate(sample["kw_label"], "keyword rules (baseline)"),
    evaluate(sample["llm_label"], f"LLM · {LABEL_SOURCE}"),
]).set_index("method")

print(f"Evaluated on {len(sample):,} problems\n")
print((scores * 100).round(1).to_string())

llm, base = scores.iloc[2], scores.iloc[1]
print(f"\nLLM vs keyword baseline:")
print(f"  hit rate    {base.hit_rate:.1%} -> {llm.hit_rate:.1%}   "
      f"({llm.hit_rate - base.hit_rate:+.1%})")
print(f"  primary acc {base.primary_acc:.1%} -> {llm.primary_acc:.1%}   "
      f"({llm.primary_acc - base.primary_acc:+.1%})")
print(f"  macro F1    {base.macro_f1:.1%} -> {llm.macro_f1:.1%}   "
      f"({llm.macro_f1 - base.macro_f1:+.1%})")

In [ ]:
# Per-topic breakdown against the strict primary label. `zero_division=0` keeps
# topics with no predictions from blowing up rather than silently vanishing.
present = sorted(set(sample["gt_primary"]) | (set(sample["llm_label"]) - {UNKNOWN}))
print(f"Per-topic report — LLM vs gt_primary\n")
print(classification_report(sample["gt_primary"], sample["llm_label"],
                            labels=present, zero_division=0))

In [ ]:
# --- Chart: which topics does the model actually get right? ------------------
# Form: ranked magnitude, one series -> horizontal bars sorted by score.
# Support (n) is shown as a direct label so a 100% on 3 problems cannot mislead.
per_topic = []
for topic in TOPICS:
    mask = sample["gt_primary"] == topic
    if mask.sum() == 0:
        continue
    per_topic.append({
        "topic": topic,
        "recall": np.mean([p in s for p, s in
                           zip(sample.loc[mask, "llm_label"], sample.loc[mask, "gt_topics"])]),
        "n": int(mask.sum()),
    })
per_topic = pd.DataFrame(per_topic).sort_values("recall")

fig, ax = plt.subplots(figsize=(8, 5.5))
y = np.arange(len(per_topic))
ax.barh(y, per_topic["recall"], height=0.62, color=SERIES[0])

overall = scores.iloc[2]["hit_rate"]
ax.axvline(overall, color=SERIES[1], linewidth=2, zorder=3)
# Headroom above the top bar, so the reference label sits INSIDE the plot rather
# than colliding with the subtitle.
ax.set_ylim(-0.75, len(per_topic) - 0.1)
ax.text(overall + 0.008, len(per_topic) - 0.2, f"overall {overall:.0%}",
        color=SERIES[1], fontsize=9, va="top")

ax.set_yticks(y)
ax.set_yticklabels(per_topic["topic"])
for yi, (r, n) in enumerate(zip(per_topic["recall"], per_topic["n"])):
    ax.text(r + 0.012, yi, f"{r:.0%}  (n={n})", va="center", fontsize=8.5, color=INK_2)

ax.set_xlim(0, 1.18)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
style_axes(ax, xgrid=True, ygrid=False)
ax.set_xlabel("hit rate — prediction is among the problem's true topics")
titled(ax, "LLM accuracy by true topic",
       f"{LABEL_SOURCE} · {len(sample):,} problems · title text only, no description")
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: confusion matrix -------------------------------------------------
# Form: magnitude over a 2-D category grid -> heatmap with a SEQUENTIAL single-hue
# ramp (light = few, dark = many). Never a rainbow: rainbow implies polarity that
# a count does not have.
# Rows are normalised so a 30%-of-data topic does not visually swamp a 2% one.
# UNKNOWN predictions are excluded so every row genuinely sums to 100%.
cm_rows = sample[sample["llm_label"] != UNKNOWN]
cm_labels = [t for t in TOPICS if (cm_rows["gt_primary"] == t).any()]
cm = confusion_matrix(cm_rows["gt_primary"], cm_rows["llm_label"], labels=cm_labels)
cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(cm_norm, cmap=SEQ_BLUE, vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(cm_labels)))
ax.set_yticks(range(len(cm_labels)))
ax.set_xticklabels(cm_labels, rotation=45, ha="right", fontsize=8.5)
ax.set_yticklabels(cm_labels, fontsize=8.5)
ax.tick_params(colors=INK_2, length=0)

# Annotate only cells worth reading; ink on 196 near-empty cells is noise.
for i in range(len(cm_labels)):
    for j in range(len(cm_labels)):
        if cm_norm[i, j] >= 0.08:
            ax.text(j, i, f"{cm_norm[i, j]:.0%}", ha="center", va="center", fontsize=7.5,
                    color="#ffffff" if cm_norm[i, j] > 0.5 else INK)

# 2px surface gap between cells — the spacer rule, so adjacent cells stay distinct.
ax.set_xticks(np.arange(len(cm_labels) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(cm_labels) + 1) - 0.5, minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2)
ax.tick_params(which="minor", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("share of the true topic's problems", color=INK_2, fontsize=9)
cbar.ax.tick_params(colors=INK_2, length=0, labelsize=8)
cbar.outline.set_visible(False)

ax.set_xlabel("predicted topic", color=INK_2)
ax.set_ylabel("true primary topic", color=INK_2)
titled(ax, "Where the model confuses topics",
       "rows sum to 100% · a strong diagonal is the goal; bright off-diagonal cells name the failure mode")
plt.tight_layout()
plt.show()

In [ ]:
# --- Read the failures -------------------------------------------------------
# Aggregate metrics hide *why* something is wrong. The most common confusions tell
# you whether the taxonomy is ambiguous or the model is actually mistaken.
hit_mask = np.array([p in s for p, s in zip(sample["llm_label"], sample["gt_topics"])])
errors = sample[~hit_mask]
print(f"{len(errors):,} misses out of {len(sample):,} ({len(errors)/len(sample):.1%})\n")

print("Most common confusions (true -> predicted):")
confusions = Counter(zip(errors["gt_primary"], errors["llm_label"]))
for (true_topic, pred), n in confusions.most_common(10):
    print(f"  {n:4d}  {true_topic:<30} -> {pred}")

print("\nSample misses:")
for row in errors.sample(min(10, len(errors)), random_state=SEED).itertuples():
    print(f"\n  {row.Title}")
    print(f"     predicted : {row.llm_label}")
    print(f"     true tags : {row.Topics}")

## 12. Do the clusters agree with the topics?

The clustering never saw a label. If cluster membership predicts topic anyway, then titles
alone carry real topical signal — which is exactly what makes the label-propagation trick in
section 14 work.

- **ARI** — agreement between two partitions, corrected for chance. 0 = random.
- **NMI** — shared information between the two partitions. 1 = identical.
- **Purity** — the share of problems that fall in a cluster whose majority topic is their own.

Expect *modest* ARI/NMI: `k = √n` gives 60 clusters against 14 topics, so a topic is
necessarily split across many clusters. Purity is the number that matters for propagation.

In [ ]:
scored = sample.copy()      # rows with both a cluster id and ground truth

ari = adjusted_rand_score(scored["gt_primary"], scored["cluster_id"])
nmi = normalized_mutual_info_score(scored["gt_primary"], scored["cluster_id"])

# Purity: for each cluster, how many members share the cluster's majority topic.
correct = 0
for cluster, group in scored.groupby("cluster_id"):
    correct += group["gt_primary"].value_counts().iloc[0]
purity = correct / len(scored)

print(f"clusters vs true topics ({len(scored):,} problems)")
print(f"  Adjusted Rand Index   : {ari:.3f}")
print(f"  Normalised Mutual Info: {nmi:.3f}")
print(f"  Cluster purity        : {purity:.1%}")
print(f"\n  (baseline purity if every problem got the majority class: "
      f"{(scored['gt_primary'] == MAJORITY_CLASS).mean():.1%})")

# Same three numbers against the LLM's labels: high agreement here means the model
# and the clustering are finding the same structure, independently.
print(f"\nclusters vs LLM labels")
print(f"  Adjusted Rand Index   : {adjusted_rand_score(scored['llm_label'], scored['cluster_id']):.3f}")
print(f"  Normalised Mutual Info: {normalized_mutual_info_score(scored['llm_label'], scored['cluster_id']):.3f}")

## 13. KNN classifier trained on the LLM's labels

The payoff of having a good vector space: once *some* problems are labelled, a
`KNeighborsClassifier` can label the rest for free.

We train on 70% of the LLM-labelled problems and test on the held-out 30%, scoring against the
**real** tags — so this measures the whole pipeline (LLM labels + KNN propagation), not just
how well KNN mimics the LLM.

`k = √n` again, with **distance weighting** so a near-identical title outvotes a distant one.

In [ ]:
# Drop unresolved rows: training a classifier on UNKNOWN teaches it to output UNKNOWN.
trainable = sample[sample["llm_label"] != UNKNOWN]
row_positions = df.index.get_indexer(trainable.index)
X_trainable = X[row_positions]

# `.to_numpy()` not `.values`: on pandas 3.0 a str column's `.values` is an
# ArrowStringArray, which sklearn cannot index with an integer array.
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X_trainable, trainable["llm_label"].to_numpy(), trainable.index.to_numpy(),
    test_size=0.30, random_state=SEED, stratify=None,
)

# k is capped at the training-set size — with a small sample, sqrt(n) can exceed it.
k_clf = min(K, len(y_tr) - 1)
knn_clf = KNeighborsClassifier(n_neighbors=k_clf, metric="cosine", weights="distance")
knn_clf.fit(X_tr, y_tr)
y_pred = knn_clf.predict(X_te)

# Index back into `sample`, not `df` — only `sample` carries the llm_label column.
test_rows = sample.loc[idx_te]
knn_hit = np.mean([p in s for p, s in zip(y_pred, test_rows["gt_topics"])])
knn_primary = accuracy_score(test_rows["gt_primary"], y_pred)
agreement = accuracy_score(y_te, y_pred)      # how faithfully KNN reproduces the LLM

print(f"train {len(y_tr):,} · test {len(y_te):,} · k = {k_clf}\n")
print(f"KNN vs LLM labels (fidelity)     : {agreement:.1%}")
print(f"KNN vs true topic set (hit rate) : {knn_hit:.1%}")
print(f"KNN vs true primary topic        : {knn_primary:.1%}")
print(f"LLM itself on the same test rows : "
      f"{np.mean([p in s for p, s in zip(test_rows['llm_label'], test_rows['gt_topics'])]):.1%}")
print(f"\nInterpretation: the gap between KNN fidelity and LLM accuracy is what "
      f"propagation costs you.")

## 14. Label efficiency — how few LLM calls do we actually need?

The practical question. Labelling all 3.6k problems costs 3.6k classifications. But if titles
cluster well, we could label **m problems per cluster**, take a majority vote, and stamp that
topic on every other member — spending only `m × k` calls.

We compare two propagation strategies at each budget:

- **Cluster vote** — majority LLM label within each k-means cluster, applied to all members.
- **KNN classifier** — `k = √n` nearest-neighbour vote over the same seed labels.

Both are scored against the real tags, so the curve shows *end-to-end* accuracy per dollar.

In [ ]:
budgets = [1, 2, 3, 5, 8, 13]      # LLM labels purchased per cluster
usable = sample[sample["llm_label"] != UNKNOWN]
curve = []

for m in budgets:
    # --- Buy m labels from each cluster -------------------------------------
    seeds = stratified_sample(usable, m)
    if len(seeds) < 20:
        continue

    # Everything not bought is what we have to predict.
    held_out = usable.drop(index=seeds.index)
    if len(held_out) < 20:
        continue

    # --- Strategy A: cluster majority vote ----------------------------------
    vote = seeds.groupby("cluster_id")["llm_label"].agg(
        lambda s: s.value_counts().idxmax())
    voted = held_out["cluster_id"].map(vote).fillna(MAJORITY_CLASS)
    vote_hit = np.mean([p in s for p, s in zip(voted, held_out["gt_topics"])])

    # --- Strategy B: KNN classifier on the same seeds -----------------------
    seed_rows = X[df.index.get_indexer(seeds.index)]
    test_rows_X = X[df.index.get_indexer(held_out.index)]
    clf = KNeighborsClassifier(n_neighbors=min(K, len(seeds) - 1),
                               metric="cosine", weights="distance")
    clf.fit(seed_rows, seeds["llm_label"].to_numpy())
    knn_pred = clf.predict(test_rows_X)
    knn_hit_m = np.mean([p in s for p, s in zip(knn_pred, held_out["gt_topics"])])

    curve.append({"per_cluster": m, "llm_calls": len(seeds),
                  "cluster_vote": vote_hit, "knn": knn_hit_m,
                  "evaluated_on": len(held_out)})

curve = pd.DataFrame(curve)
print("Propagation accuracy vs labelling budget "
      f"(full-labelling hit rate = {scores.iloc[2]['hit_rate']:.1%})\n")
print(curve.assign(cluster_vote=lambda d: (d.cluster_vote * 100).round(1),
                   knn=lambda d: (d.knn * 100).round(1)).to_string(index=False))

In [ ]:
# --- Chart: accuracy vs labelling budget ------------------------------------
# Form: change over an ordered quantity -> line chart. Two series, both direct-labelled
# at the line end so the legend is a backup rather than the only key.
fig, ax = plt.subplots(figsize=(7.5, 5))

full = scores.iloc[2]["hit_rate"]
ax.axhline(full, color=MUTED, linewidth=1.5, linestyle=(0, (4, 3)), zorder=1)
# Label hangs BELOW the reference line — above it would run into the subtitle.
ax.text(curve["llm_calls"].iloc[0], full - 0.006,
        f"label everything ({full:.0%})", color=INK_2, fontsize=9, va="top")

# The two series converge at the right-hand end, so end-of-line direct labels would
# sit on top of each other. The legend carries identity here instead.
for column, colour, label in [("cluster_vote", SERIES[0], "cluster majority vote"),
                              ("knn", SERIES[1], f"KNN classifier (k={K})")]:
    ax.plot(curve["llm_calls"], curve[column], color=colour, linewidth=2,
            marker="o", markersize=7, markeredgecolor=SURFACE, markeredgewidth=1.5,
            label=label, zorder=3)

style_axes(ax, ygrid=True)
ax.set_xlabel("LLM calls spent (seed labels)")
ax.set_ylabel("hit rate on the unlabelled remainder")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlim(0, curve["llm_calls"].max() * 1.12)
legend = ax.legend(loc="lower right", frameon=False, fontsize=9)
for text in legend.get_texts():
    text.set_color(INK_2)
titled(ax, "Accuracy per labelling budget",
       f"seed m problems per cluster, propagate to the rest · k = sqrt(n) = {K}")
plt.tight_layout()
plt.show()

## 15. Summary and export

In [ ]:
# Attach every derived column back onto the full frame and write it out, so the
# clusters and labels are reusable without re-running the notebook.
df["llm_label"] = df["Title"].map(label_map)
df["kw_label"] = df["Title"].map(keyword_label)
df["cluster_terms"] = df["cluster_id"].map(
    lambda c: ", ".join(cluster_top_terms(c, 4)))

export_cols = ["ID", "Title", "Difficulty", "Topics", "Category",
               "cluster_id", "cluster_terms", "gt_primary", "llm_label", "kw_label"]
out_path = OUT_DIR / "leetcode_clustered_labelled.csv"
df[export_cols].to_csv(out_path, index=False)

scores.to_csv(OUT_DIR / "accuracy_scores.csv")
curve.to_csv(OUT_DIR / "label_efficiency.csv", index=False)

print(f"wrote {out_path}  ({len(df):,} rows)")
print(f"wrote {OUT_DIR / 'accuracy_scores.csv'}")
print(f"wrote {OUT_DIR / 'label_efficiency.csv'}")
print(f"cache: {CACHE_PATH}  ({len(load_cache()):,} cached labels)")

In [ ]:
# =============================================================================
# Headline numbers
# =============================================================================
llm_row = scores.iloc[2]
best_budget = curve.loc[curve["cluster_vote"].idxmax()] if len(curve) else None

print("=" * 74)
print("  RESULTS")
print("=" * 74)
print(f"  dataset            {N:,} problems · {len(labelled):,} with tags")
print(f"  k = sqrt(n)        {K}  (neighbours, clusters and classifier all share it)")
print(f"  clustering         silhouette {sil:.3f} · purity {purity:.1%} · NMI {nmi:.3f}")
print("-" * 74)
print(f"  labels from        {LABEL_SOURCE}")
print(f"  evaluated on       {len(sample):,} problems")
print(f"  hit rate           {llm_row['hit_rate']:.1%}   "
      f"(vs {scores.iloc[1]['hit_rate']:.1%} keyword, "
      f"{scores.iloc[0]['hit_rate']:.1%} majority)")
print(f"  primary accuracy   {llm_row['primary_acc']:.1%}")
print(f"  macro F1           {llm_row['macro_f1']:.1%}")
print("-" * 74)
if best_budget is not None:
    saving = 1 - best_budget["llm_calls"] / len(labelled)
    print(f"  cheapest good run  {int(best_budget['llm_calls'])} LLM calls "
          f"-> {best_budget['cluster_vote']:.1%} on the remaining "
          f"{int(best_budget['evaluated_on']):,}")
    print(f"                     ({saving:.0%} fewer calls than labelling everything)")
print("=" * 74)

### What to try next

- **Richer input.** We classified from the *title alone*. Feeding the problem description
  should push hit rate well past what a 5-word title can support — the honest ceiling here
  is low because the input is thin.
- **Sentence embeddings instead of TF-IDF.** `sentence-transformers/all-MiniLM-L6-v2` would
  put `Number of Islands` near `Max Area of Island` even with zero shared tokens.
- **Sweep k.** `√n` is a starting point, not an answer. Plot silhouette and purity over
  `k ∈ [10, 200]` and see where purity actually peaks.
- **Multi-label.** Real problems carry 3.5 tags on average. Ask the model for its top 3 and
  score with precision@3 instead of forcing a single choice.
- **Self-consistency.** Run each batch 3× at `temperature=0.7` and take the majority vote;
  disagreement across runs is a useful confidence signal for triage.